# 线性回归练习题

在这个练习中，我们使用一个Kaggle竞赛中提供的共享单车的数据集：[Bike Sharing Demand](https://www.kaggle.com/c/bike-sharing-demand/data)。
该数据集包含2011到2012年Capital Bikeshare系统中记录的每日每小时单车的租赁数，以及相应的季节和气候等信息。

数据列：
* **datetime** - hourly date + timestamp  
* **season** -  1 = spring, 2 = summer, 3 = fall, 4 = winter 
* **holiday** - whether the day is considered a holiday
* **workingday** - whether the day is neither a weekend nor holiday
* **weather** - 1: Clear, Few clouds, Partly cloudy, Partly cloudy；2: Mist + Cloudy, Mist + Broken clouds, Mist + Few clouds, Mist；3: Light Snow, Light Rain + Thunderstorm + Scattered clouds, Light Rain + Scattered clouds；4: Heavy Rain + Ice Pallets + Thunderstorm + Mist, Snow + Fog 
* **temp** - temperature in Celsius
* **atemp** - "feels like" temperature in Celsius
* **humidity** - relative humidity
* **windspeed** - wind speed
* **casual** - number of non-registered user rentals initiated
* **registered** - number of registered user rentals initiated
* **count** - number of total rentals

## 第一步：读入数据

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn import metrics
import numpy as np

# 1. 读取数据
# 注意：请确保文件路径正确。如果文件就在当前目录下，直接写 'bikeshare.csv' 即可
try:
    bikes = pd.read_csv('bikeshare.csv', index_col='datetime', parse_dates=True)
    print("数据读取成功！")
except FileNotFoundError:
    print("找不到文件，请检查文件路径是否正确。")
  

# 查看数据前几行
# print(bikes.head())

In [ ]:
!pip install seaborn

In [ ]:
bikes.head()

## 第二步：可视化数据

* 用matplotlib画出温度“temp”和自行车租赁数“count”之间的散点图；
* 用seborn画出温度“temp”和自行车租赁数“count”之间带线性关系的散点图（提示：使用seaborn中的lmplot绘制）

In [ ]:
# matplotlib
# 1. 使用 matplotlib 画散点图
plt.scatter(bikes['temp'], bikes['count'])
plt.xlabel('Temperature')
plt.ylabel('Count')
plt.title('Matplotlib: Temp vs Count')
plt.show()

In [ ]:
# seaborn
import seaborn as sns

# 使用 seaborn lmplot 绘制带线性关系的散点图
sns.lmplot(x='temp', y='count', data=bikes, aspect=1.5)
plt.title('Seaborn: Temp vs Count with Linear Fit')
plt.show()

## 第三步：一元线性回归

用温度预测自行车租赁数

In [ ]:
# create X and y
# 1. 创建 X (特征) 和 y (目标变量)
feature_cols = ['temp']
X = bikes[feature_cols]
y = bikes['count']

In [ ]:
# import, instantiate, fit
# 2. 实例化并训练模型
linreg = LinearRegression()
linreg.fit(X, y)


In [ ]:
# print the coefficients
# 3. 打印系数 (截距和斜率)
print("一元线性回归系数:")
print(f"截距 (Intercept): {linreg.intercept_}")
print(f"系数 (Coefficient): {linreg.coef_}")

## 第四步：探索多个特征

In [ ]:
# explore more features
feature_cols = ['temp', 'season', 'weather', 'humidity']

In [ ]:
# using seaborn, draw multiple scatter plots between each feature in feature_cols and 'count'
# 使用 seaborn 绘制每个特征与 count 的散点图
import seaborn as sns
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for idx, col in enumerate(feature_cols):
    ax = axes[idx // 2][idx % 2]
    sns.scatterplot(x=bikes[col], y=bikes['count'], ax=ax, alpha=0.3)
    ax.set_xlabel(col)
    ax.set_ylabel('count')
    ax.set_title(f'{col} vs count')
plt.tight_layout()
plt.show()

# 计算相关系数矩阵
corr_matrix = bikes[feature_cols + ['count']].corr()
print("\n相关系数矩阵:")
print(corr_matrix)

In [ ]:
# correlation matrix (ranges from 1 to -1)
bikes.corr()

In [ ]:
sns.heatmap(bikes.corr())

### 用'temp', 'season', 'weather', 'humidity'四个特征预测单车租赁数'count'

In [ ]:
# create X and y
X = bikes[feature_cols]
y = bikes['count']

In [ ]:
# import, instantiate, fit
linreg_multi = LinearRegression()
linreg_multi.fit(X, y)

In [ ]:
# print the coefficients
print("\n多元线性回归系数:")
for name, coef in zip(feature_cols, linreg_multi.coef_):
    print(f"{name}: {coef}")

### 使用train/test split和RMSE来比较多个不同的模型

In [ ]:
# compare different sets of features
feature_cols1 = ['temp', 'season', 'weather', 'humidity']
feature_cols2 = ['temp', 'season', 'weather']
feature_cols3 = ['temp', 'season', 'humidity']

# 使用 train/test split 和 RMSE 来比较不同的特征组合
from sklearn.model_selection import train_test_split
from sklearn import metrics

# 划分训练集和测试集（使用第一种特征组合来划分，保证各模型用同样的划分）
X_train, X_test, y_train, y_test = train_test_split(bikes[feature_cols1], bikes['count'], 
                                                      test_size=0.25, random_state=42)

for name, cols in [('Feature Set 1', feature_cols1), 
                    ('Feature Set 2', feature_cols2), 
                    ('Feature Set 3', feature_cols3)]:
    X_tr = X_train[cols]
    X_te = X_test[cols]
    lr = LinearRegression()
    lr.fit(X_tr, y_train)
    y_pred = lr.predict(X_te)
    rmse = np.sqrt(metrics.mean_squared_error(y_test, y_pred))
    print(f"{name} {cols}: RMSE = {rmse:.4f}")

## 补充：处理类别特征

有两种类别特征：

- **有序类别值：** 转换成相应的数字值(例如: small=1, medium=2, large=3)
- **无序类别值:** 使用dummy encoding (0/1编码)

此数据集中的类别特征有：

- **有序类别值：** weather (已经被编码成相应的数字值1,2,3,4)
- **无序类别值：** season (需要进行dummy encoding), holiday (已经被dummy encoded), workingday (已经被dummy encoded)

In [ ]:
# create dummy variables
season_dummies = pd.get_dummies(bikes.season, prefix='season')

# print 5 random rows
season_dummies.sample(n=5, random_state=1)

我们只需要 **三个 dummy 变量 (不是四个)** （为什么？）, 所以可以删除第一个dummy变量。

In [ ]:
# drop the first column
season_dummies.drop(season_dummies.columns[0], axis=1, inplace=True)

# print 5 random rows
season_dummies.sample(n=5, random_state=1)

In [ ]:
# concatenate the original DataFrame and the dummy DataFrame (axis=0 means rows, axis=1 means columns)
bikes = pd.concat([bikes, season_dummies], axis=1)

# print 5 random rows
bikes.sample(n=5, random_state=1)

### 将编码成的dummy变量加入回归模型的特征，预测单车租赁数

In [ ]:
# include dummy variables for season in the model
feature_cols = ['temp', 'season_2', 'season_3', 'season_4', 'humidity']

# 使用包含dummy变量的特征训练模型，并与前面模型比较
X_dummy = bikes[feature_cols]
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_dummy, bikes['count'], 
                                                              test_size=0.25, random_state=42)
lr_dummy = LinearRegression()
lr_dummy.fit(X_train_d, y_train_d)
y_pred_dummy = lr_dummy.predict(X_test_d)
rmse_dummy = np.sqrt(metrics.mean_squared_error(y_test_d, y_pred_dummy))
print(f"含dummy变量的模型 {feature_cols}: RMSE = {rmse_dummy:.4f}")

# 打印系数
print("\n含dummy变量模型的回归系数:")
for name, coef in zip(feature_cols, lr_dummy.coef_):
    print(f"  {name}: {coef:.4f}")
print(f"  截距: {lr_dummy.intercept_:.4f}")

和前面的模型进行比较